# B2-Li 760 r8: дотюн plain/nonunit
Старт из EMA best.pt рана disentangle_b2_li760_r8_long; новый optimizer/scheduler.
3 × 24000 примеров. Все положительные — plain/nonunit, 25% draws — отрицательные всех доменов. Общая development-валидация и выбор best по общей метрике сохранены.
LR encoder 1e-5, JPEG/head 3e-5. Полные кадры, без JPEG-пережатия. Архитектура и размер 760 сохранены.
На сервере настройте .env: batch 16, accumulation 1. Последняя ячейка запускает обучение.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

experiment = 'experiments/disentangle_b2_li760_r8_plain_nonunit_ft'
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1080, 1920)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
from src.training.domain_focus import PlainNonunitFocus
from src.data.data_workspace import DataWorkspace
checkpoint = cfg.paths.runs_path / cfg.train.finetune_from
assert checkpoint.is_file(), f"Missing source checkpoint: {checkpoint}"
focus = PlainNonunitFocus.select(protocol.rows("train"), DataWorkspace(cfg.paths.data_path).train_root, cfg.train.workers)
print("Focus rows:", len(focus), "positive:", (~focus.is_negative).sum(), "negative:", focus.is_negative.sum())


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
